> 历史对照：本 Notebook 固定使用 part2-20260924-v1 旧权重；当前正式模型已加入 SST 排序监督，见 Notebook 01 和 docs/SST_TRAINING.md。

# 还原态测试肽：结构与已有 Cys 位点评分复核

本案例对应反馈意见（3）：对 Cys3 和 Cys14 评分，不插入残基。实验排序已知为 Cys3 效率更高；当前计算未稳定复现。默认重新提取随附全部结构的特征并评分，不重新训练或启动分子动力学。

已确认 pH 7.5、300 K、Ac/NH2 端基、Cys 巯基无保护；TFA 浓度与缓冲体系未知，模拟以 NaCl 水溶液近似。详见 docs/REDUCED_PEPTIDE_CASE.md。

In [1]:
from pathlib import Path
import os, sys, json, csv, subprocess, hashlib, html
from uuid import uuid4
from IPython.display import display, HTML
start=Path(os.environ.get("CYSRANK_ROOT",Path.cwd())).resolve()
ROOT=next((p for p in [start,*start.parents] if (p/"tools/analyze_2mi1_reduced_md.py").is_file()),None)
assert ROOT is not None, "从项目内启动，或设置 CYSRANK_ROOT"
CASE=ROOT/"results/2mi1_reduced_capped_ph75_20260924"
def read(p): return json.loads(p.read_text(encoding="utf-8-sig"))
def table(rows):
    keys=list(rows[0])
    text="<table><tr>"+"".join("<th>"+html.escape(k)+"</th>" for k in keys)+"</tr>"
    for r in rows:text+="<tr>"+"".join("<td>"+html.escape(str(r[k]))+"</td>" for k in keys)+"</tr>"
    display(HTML(text+"</table>"))
protocol=read(CASE/"protocol.json"); summary=read(CASE/"analysis/summary.json")
assert protocol["status"]=="complete" and protocol["preparation_pH"]==7.5
assert hashlib.sha256((ROOT/"logs/retrain_20260924/final_model/site_predictor.pt").read_bytes()).hexdigest()==protocol["checkpoint_sha256"]
table([{"起点":protocol["starts"],"pH":protocol["preparation_pH"],"温度K":protocol["temperature_K"],"每起点生产ps":protocol["production_ps_per_start"],"OpenMM":protocol["openmm"]}])

起点,pH,温度K,每起点生产ps,OpenMM
10,7.5,300.0,100,8.6.1


## 实际结果

100 个生产快照来自 10 条短轨迹，有时间相关性；下表不表示独立样本统计或反应效率。结构已解除二硫键并松弛，但不能声称构象采样收敛。

In [2]:
s=summary["production"]
table([{"位点":"Cys3","平均原始分数":s["cys3_score"]["mean"]},{"位点":"Cys14","平均原始分数":s["cys14_score"]["mean"]}])
table([{"起点":i,"Cys3均分":r["cys3_score"]["mean"],"Cys14均分":r["cys14_score"]["mean"],"Cys3更高帧数":str(r["cys3_higher"])+"/"+str(r["frames"])} for i,r in summary["per_start"].items()])
print("全部生产快照中 Cys3 更高：",s["cys3_higher"],"/",s["frames"])
with (CASE/"analysis/results.csv").open(encoding="utf-8-sig",newline="") as f: rows=list(csv.DictReader(f))
assert len(rows)==200
assert all((ROOT/r["structure_file"]).is_file() for r in rows)
print("200 行逐位点结果的结构链接均存在。")

位点,平均原始分数
Cys3,0.8918273895978928
Cys14,0.9852838832139968


起点,Cys3均分,Cys14均分,Cys3更高帧数
1,0.8513365805149078,0.983062207698822,0/10
2,0.9051324009895325,0.9783635556697845,1/10
3,0.8363015234470368,0.9883765280246735,0/10
4,0.947878235578537,0.996241670846939,0/10
5,0.8880827188491821,0.9798671722412109,0/10
6,0.9195508003234864,0.9953725636005402,0/10
7,0.9563833475112915,0.9743527352809906,2/10
8,0.8105174899101257,0.9946325480937958,0/10
9,0.8335606813430786,0.9839780926704407,0/10
10,0.9695301175117492,0.9785917580127717,4/10


全部生产快照中 Cys3 更高： 7 / 100
200 行逐位点结果的结构链接均存在。


## 从结构重新提取特征并评分

使用正式入口重算所有最小化、平衡及生产结构；不会挑选有利构象。使用当前内核解释器，不依赖 tmp 中的代码或运行库。输出写入新的 logs/notebook 子目录，正式结果不覆盖。

In [3]:
RUN=ROOT/"logs/notebook"/("reduced_case_"+uuid4().hex[:8])
command=[sys.executable,"-B",str(ROOT/"tools/analyze_2mi1_reduced_md.py"),"--output",str(RUN)]
done=subprocess.run(command,cwd=ROOT,capture_output=True,text=True,encoding="utf-8",errors="replace")
if done.returncode: raise RuntimeError(done.stdout+done.stderr)
(RUN/"replay.log").write_text(done.stdout+done.stderr,encoding="utf-8")
replay=read(RUN/"summary.json")
assert replay["production"]==summary["production"]
assert replay["per_start"]==summary["per_start"]
assert (RUN/"scores.jsonl").read_bytes()==(CASE/"analysis/scores.jsonl").read_bytes()
assert (RUN/"results.csv").read_bytes()==(CASE/"analysis/results.csv").read_bytes()
print("240 个位点评分、200 行标准化清单及逐起点统计与正式结果一致。")
print("重算输出：",RUN.relative_to(ROOT).as_posix())

240 个位点评分、200 行标准化清单及逐起点统计与正式结果一致。
重算输出： logs/notebook/reduced_case_db4e01fd


## 结论与边界

在这轮状态更匹配的近似下，模型仍主要给 Cys14 更高分，未稳定复现实验排序。模型接触过相关训练片段；这是案例复核，不是独立有效性验证。有限采样、TFA/缓冲近似、固定质子化及端基未进入序列编码等限制均须保留。

重新做完整模拟的命令见 docs/REDUCED_PEPTIDE_CASE.md，需要 requirements-md.txt 与 OpenCL 驱动。本 Notebook 默认只复算结构特征和分数，避免每次展示都重新生成轨迹；原模拟参数、种子、能量和执行脚本在 logs/reduced_md_20260924。